# Equivalence-ratio (fuel-flow) combustion instability

Many combustors go unstable through their **fuel supply**.  Fuel is injected upstream of the
flame; a chamber fluctuation modulates the **fuel flow rate**, which changes the local
**equivalence ratio**; that mixture fluctuation **convects** to the flame and is burned into an
unsteady heat release, with the time lag of the injector-to-flame transport.  If that lag aligns
with an acoustic mode, the mode grows.

The rig follows that chain: an **air-only inlet**, a **fuel injector** (a `mass_source` carrying
H2) a meaningful distance upstream, a **static** reacting flame (no prescribed flame response --
its heat release responds only through the mixture it is fed), and an outlet.  The injector's
**fuel flow rate carries an `n-tau` transfer function** -- the only dynamic element.  Because the
unsteady fuel modulates the injected mass, momentum, enthalpy *and composition*, the injector
sheds a convected **mixture-fraction wave** -- the equivalence-ratio wave the flame then burns.

We assess stability with the **Nyquist open-loop driver**
(:func:`nefes.perturbation.open_loop_response`), which works on the **real frequency axis**.  That
matters doubly here: the reacting flow's convected-composition spectrum makes the complex-plane
**eigensolver unreliable** (it cannot certify its count), and the fuel feed could equally be a
*measured* transfer function -- both are exactly the cases the real-axis driver handles.

**Two configurations.**  First a *clean* rig with an acoustically **stiff** (mass-flow) air inlet,
so the fuel-flow FTF is the **sole** equivalence-ratio driver and the baseline is provably passive.
Then, briefly, the configuration with a **total-pressure (plenum) air inlet**, where the air flow
*itself* fluctuates and co-drives the instability -- realistic, but the cause is no longer the fuel
feed alone.

In [ ]:
import os, sys, warnings

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from nefes.thermo import SpeciesLibrary, Thermo

from nefes.elements import catalog as cat
from nefes.shell import build_problem
from nefes.elements.dynamic_source import mass_flow_response, n_tau_lowpass
from nefes.perturbation.operator.boundary_bc import PerturbationBC
from nefes.perturbation import open_loop_response, eigenmodes, forced_response
from nefes.solver import solve
from nefes.solver.report import states_table
from nefes.shell import Network
from nefes.thermo.api import EQ_FROZEN, EQ_KERNEL
from nefes.thermo.configure import equilibrium
from nefes.assembly.recover import ES_T, ES_M, ES_U
from nefes.plotting import use_nefes_theme

use_nefes_theme()
MECH = os.path.join(_root, "nefes", "thermo", "data", "h2o2.yaml")
AIR = {"O2": 0.21, "N2": 0.79}
AREA = 0.02
MDOT_AIR, MDOT_FUEL = 0.4, 0.006          # air [kg/s], H2 [kg/s]  -> lean (phi ~ 0.5)
La, Lb, Lc = 0.4, 0.5, 0.5                 # inlet->injector, injector->flame, flame->outlet [m]
E_AIR, E_INJ_OUT, E_APPROACH, E_FLAME_OUT = 0, 2, 3, 4

lib = SpeciesLibrary.from_cantera(MECH); gas = Thermo(lib); _idx = lib.species_index
_Yair = np.zeros(lib.n_species); _Yair[_idx["O2"]] = 0.21; _Yair[_idx["N2"]] = 0.79; _Yair /= _Yair.sum()
H_AIR = gas.enthalpy_mass(_Yair, 300.0)

def fuel_ftf(n, tau, fc):
    "Fuel flow rate responds to the acoustic velocity at the injector outlet, with lag + roll-off."
    return mass_flow_response(n_tau_lowpass(n, tau, fc), ref_edge=E_INJ_OUT, quantity="u")

## 1. The clean rig: a stiff air inlet

A **mass-flow inlet** holds the air flow fixed acoustically ($\dot m' = 0$), so the air cannot
modulate the equivalence ratio -- only the fuel-flow FTF can.  A **pressure outlet** (open end)
sets the pressure level and radiates acoustic energy (damping).  With no fuel FTF this network has
no active feedback and is **passive**; the fuel FTF is the sole driver.

In [ ]:
def build_clean(n=3.5, tau=2.0e-3, fc=300.0, Lb_=Lb, dynamic=True, inlet_excite=False):
    ds = fuel_ftf(n, tau, fc) if dynamic else None
    inlet_bc = PerturbationBC.anechoic(driven=("acoustic",)) if inlet_excite else PerturbationBC.inherit()
    els = [
        cat.mass_flow_inlet(MDOT_AIR, 300.0, composition=AIR, basis="mole", perturbation_bc=inlet_bc),
        cat.duct(La),
        cat.mass_source(MDOT_FUEL, 300.0, composition={"H2": 1.0}, basis="mole", dynamic_source=ds),
        cat.duct(Lb_),
        cat.equilibrium_flame(),                       # static: no prescribed flame response
        cat.duct(Lc),
        cat.pressure_outlet(1.0e5, Tt_backflow=300.0, composition=AIR, basis="mole",
                            perturbation_bc=PerturbationBC.open_end()),
    ]
    edges = [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA), (4, 5, AREA), (5, 6, AREA)]
    edge_models = [EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]
    prob = build_problem(equilibrium(gas.mech), els, edges,
                             mdot_ref=MDOT_AIR, p_ref=1e5, h_ref=H_AIR, edge_models=edge_models)
    network = Network(equilibrium(gas.mech), p_ref=1e5, mdot_ref=MDOT_AIR, h_ref=H_AIR,
                      nodes=els, edges=edges, edge_models=edge_models)
    return prob, network

prob, network = build_clean()
x = solve(prob).x
est = states_table(prob, x)
u_app = float(est[ES_U, E_APPROACH])
print(f"equivalence ratio phi ~ {(MDOT_FUEL / MDOT_AIR) / 0.0292:.2f} (lean)")
print(f"flame: T = {est[ES_T, E_FLAME_OUT]:.0f} K,  M = {est[ES_M, E_FLAME_OUT]:.3f}")
print(f"injector -> flame convective lag  Lb/u = {Lb / u_app * 1e3:.1f} ms")

## 2. The mechanism: a fuel pulse becomes a convected mixture wave

Drive the air inlet with a unit acoustic wave (real frequencies).  The fuel feed fluctuates, so
the **injected fuel fraction** fluctuates -- only *downstream* of the injector (no fuel upstream)
-- convects to the flame conserving its magnitude (phase $e^{-i\omega L_b/u}$), and the static
flame burns it into an unsteady heat release (the entropy wave $h$ it sheds).

In [ ]:
prob_exc, _ = build_clean(dynamic=True, inlet_excite=True)   # drive the air inlet with a unit acoustic wave
ns = int(prob_exc.n_solve)
s_fuel = ns - 1                                          # injected ("source") stream fraction
fsw = np.linspace(60.0, 800.0, 260)
fr = forced_response(prob_exc, x, fsw, isentropic=False)
z_air = np.abs(fr.X[:, ns * E_AIR + s_fuel])
z_inj = np.abs(fr.X[:, ns * E_INJ_OUT + s_fuel])
z_flame = np.abs(fr.X[:, ns * E_APPROACH + s_fuel])
h_flame = np.abs(fr.waves(E_FLAME_OUT)[:, 2])

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "fuel-fraction wave |Z'| around the injector", "heat-release fluctuation (entropy |h|) at the flame"))
fig.add_scatter(x=fsw, y=z_air, name="air approach (upstream)", row=1, col=1)
fig.add_scatter(x=fsw, y=z_inj, name="injector outlet", row=1, col=1)
fig.add_scatter(x=fsw, y=z_flame, name="flame approach", row=1, col=1, line=dict(dash="dot"))
fig.add_scatter(x=fsw, y=h_flame, name="flame outlet |h|", row=1, col=2)
fig.update_xaxes(title_text="frequency [Hz]", row=1, col=1); fig.update_xaxes(title_text="frequency [Hz]", row=1, col=2)
fig.update_layout(title="Fuel flow -> mixture-fraction wave -> convect -> flame heat release")
fig.show()
print("no fuel-fraction fluctuation upstream of the injector: max |Z'_air| =", f"{z_air.max():.2e}")
print("lossless convection injector->flame: max |Z'_inj - Z'_flame| =", f"{np.max(np.abs(z_inj - z_flame)):.2e}")

## 3. Stability: the fuel-flow return ratio

The injector's fuel-flow FTF is the network's only dynamic source, so its **return ratio**
$L(\omega)$ is what Nyquist tracks.  Keeping the convected mixture wave (`isentropic=False`) the
locus **encircles the critical point $+1$** -- the fuel-flow coupling drives a mode unstable.
Freezing the convected wave (`isentropic=True`) removes the equivalence-ratio lag and the
encirclement vanishes: the **convective transport is the destabilizing link**.

In [ ]:
freqs = np.linspace(0.0, 1500.0, 1100)
nyq_full = open_loop_response(prob, x, freqs, isentropic=False)
nyq_iso = open_loop_response(prob, x, freqs, isentropic=True)
print("convected mixture wave retained : n_unstable =", nyq_full.n_unstable, " margin =", round(nyq_full.margin, 3),
      " crossing ~", [round(c["freq_hz"], 1) for c in nyq_full.crossings(0.35)], "Hz")
print("mixture wave frozen (isentropic): n_unstable =", nyq_iso.n_unstable, " margin =", round(nyq_iso.margin, 3))

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "frozen mixture wave -- no encirclement", "convected mixture wave -- encircles +1"))
for col, r in ((1, nyq_iso), (2, nyq_full)):
    fig.add_scatter(x=r.L.real, y=r.L.imag, mode="lines", line=dict(color="#1f77b4"),
                    name="L(w), w>0", showlegend=(col == 1), row=1, col=col)
    fig.add_scatter(x=r.L.real, y=-r.L.imag, mode="lines", line=dict(color="#1f77b4", dash="dot"),
                    name="L(w), w<0", showlegend=(col == 1), row=1, col=col)
    fig.add_scatter(x=[1.0], y=[0.0], mode="markers", marker=dict(symbol="x", size=12, color="#d62728"),
                    name="critical +1", showlegend=(col == 1), row=1, col=col)
fig.update_xaxes(title_text="Re L", row=1, col=1); fig.update_xaxes(title_text="Re L", row=1, col=2)
fig.update_yaxes(title_text="Im L", row=1, col=1)
fig.update_layout(title="Nyquist locus of the fuel-flow return ratio L(w)")
fig.show()

## 4. Why the real-axis driver: the eigensolver is unreliable here

The reacting flow carries convected **composition** waves, so the operator's complex-plane
spectrum fills with dense, near-marginal convective modes.  The contour eigensolver cannot certify
its count (and returns an implausible number of "unstable" modes for what is a radiating,
fuel-only-driven duct).  The real-axis Nyquist count above is the reliable verdict.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    em = eigenmodes(prob, x, freq_band=(60.0, 900.0), growth_band=(-250.0, 200.0),
                    isentropic=False, residual_tol=1e-8)
n_em_unstable = int(np.sum(em.growth_rates > 0))
print(f"eigenmodes (entropy+composition retained): resolved {em.n_modes} modes, "
      f"of which {n_em_unstable} flagged 'unstable', certified = {em.certified}")
print(f"Nyquist (same operator, real axis): n_unstable = {nyq_full.n_unstable}  <-- the reliable count")

## 5. The convective lag controls stability

The instability lives in the injector-to-flame **transport lag**.  Sweeping the distance $L_b$
(which leaves the mean flow untouched -- duct length is inert in the steady solve) walks the
equivalence-ratio phase through the acoustic modes, switching stability on and off.  This
lag-driven banding is the fingerprint of an equivalence-ratio instability.

In [ ]:
Lb_scan = np.linspace(0.15, 0.95, 33)
counts, smargin = [], []
for Lb_i in Lb_scan:
    p, _ = build_clean(Lb_=Lb_i)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r = open_loop_response(p, x, freqs, isentropic=False)   # mean unchanged: reuse x
        counts.append(r.n_unstable)
        smargin.append(r.margin if r.n_unstable == 0 else -r.margin)   # sign by stability for the eye

lag_ms = Lb_scan / u_app * 1e3
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("unstable modes (fuel-flow coupling)", "signed margin (+ stable / - unstable)"))
fig.add_scatter(x=lag_ms, y=counts, mode="lines+markers", row=1, col=1, name="n_unstable")
fig.add_scatter(x=lag_ms, y=smargin, mode="lines+markers", row=2, col=1, name="signed margin")
fig.add_hline(y=0.0, line=dict(color="#d62728", dash="dash"), row=2, col=1)
fig.update_yaxes(title_text="count", row=1, col=1); fig.update_yaxes(title_text="+/- |1 - L|", row=2, col=1)
fig.update_xaxes(title_text="injector -> flame convective lag [ms]", row=2, col=1)
fig.update_layout(title="Stability vs the equivalence-ratio convective lag")
fig.show()

## 6. The plenum (total-pressure) air inlet -- a co-driven instability

Replace the stiff inlet with a **total-pressure inlet** (a plenum-fed air supply) and the outlet
with a **mass-flow outlet** -- the configuration of a combustor whose air is acoustically *soft*.
Now the **air flow itself fluctuates**, which *also* modulates the equivalence ratio (an
"air-side" coupling), and the boundaries are energy-neutral (little acoustic damping).  The
fuel-flow return ratio still encircles $+1$ (and still needs the convected mixture wave), so the
fuel coupling is destabilizing here too -- but the instability is now **co-driven** by the air
side, and the baseline is no longer cleanly passive (the eigensolver is likewise uninformative in
this undamped reacting regime).  The Nyquist return ratio isolates the **fuel-flow contribution**.

In [ ]:
def build_plenum(n=2.0, tau=1.0e-3, fc=400.0, Lb_=0.4, dynamic=True):
    ds = mass_flow_response(n_tau_lowpass(n, tau, fc), ref_edge=E_AIR, quantity="u") if dynamic else None
    els = [
        cat.total_pressure_inlet(1.2e5, 300.0, composition=AIR, basis="mole", perturbation_bc=PerturbationBC.inherit()),
        cat.duct(La),
        cat.mass_source(MDOT_FUEL, 300.0, composition={"H2": 1.0}, basis="mole", dynamic_source=ds),
        cat.duct(Lb_),
        cat.equilibrium_flame(),
        cat.duct(Lc),
        cat.mass_flow_outlet(MDOT_AIR + MDOT_FUEL, perturbation_bc=PerturbationBC.inherit()),
    ]
    edges = [(0, 1, AREA), (1, 2, AREA), (2, 3, AREA), (3, 4, AREA), (4, 5, AREA), (5, 6, AREA)]
    edge_models = [EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]
    return build_problem(equilibrium(gas.mech), els, edges,
                             mdot_ref=MDOT_AIR, p_ref=1e5, h_ref=H_AIR, edge_models=edge_models)

prob_p = build_plenum()
xp = solve(prob_p).x
fp = np.linspace(0.0, 1400.0, 900)
full_p = open_loop_response(prob_p, xp, fp, isentropic=False)
iso_p = open_loop_response(prob_p, xp, fp, isentropic=True)
print("plenum air inlet -- fuel-flow coupling contribution:")
print("  convected mixture wave retained : encirclements of +1 =", full_p.n_unstable, " margin =", round(full_p.margin, 3))
print("  mixture wave frozen (isentropic): encirclements of +1 =", iso_p.n_unstable, " margin =", round(iso_p.margin, 3))

fig = go.Figure()
fig.add_scatter(x=full_p.L.real, y=full_p.L.imag, mode="lines", name="full (mixture wave)")
fig.add_scatter(x=iso_p.L.real, y=iso_p.L.imag, mode="lines", line=dict(dash="dot"), name="frozen")
fig.add_scatter(x=[1.0], y=[0.0], mode="markers", marker=dict(symbol="x", size=12, color="#d62728"), name="critical +1")
fig.update_yaxes(scaleanchor="x", scaleratio=1.0)
fig.update_layout(title="Plenum-inlet rig: fuel-flow return ratio L(w)", xaxis_title="Re L", yaxis_title="Im L")
fig.show()

## Summary

* A **fuel-flow `n-tau`** on an upstream injector -- with a **static** flame -- drives a
  thermoacoustic instability: the fuel pulse sheds a **mixture-fraction wave** that convects to the
  flame (none upstream, magnitude conserved) and is burned into an unsteady heat release.
* With an acoustically **stiff air inlet** the fuel feed is the *sole* equivalence-ratio driver and
  the baseline is passive: the fuel-flow **return ratio encircles $+1$** (one unstable mode), and
  *only* with the convected mixture wave -- so the **equivalence-ratio convective lag** is the
  destabilizing link (the stability bands with the injector-to-flame distance).
* With a **plenum (total-pressure) air inlet** the air flow co-drives the instability; the Nyquist
  return ratio still isolates the fuel-flow contribution.
* The contour eigensolver cannot certify a count in this reacting/convective regime; the real-axis
  **Nyquist driver** gives the reliable verdict -- and would equally accept a *measured* fuel
  transfer function -- the capability this case exercises.

## Export for Nemo

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in Nemo.

In [ ]:
import os, tempfile

sol = network.solve()  # the primary clean-rig mean flow, as a shell Solution
_out = os.path.join(tempfile.mkdtemp(), "equivalence_ratio_instability.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)